# 音频在地图上的位置

在 AutoDL 的 volcalmap 目录打开此笔记本，选择 Python 3 内核并依次运行。默认查询一段 JVS 录音。要定位自己的录音，用 JupyterLab 上传单人单声道 WAV，再把下一格的 audio 改成它的路径。

In [ ]:
from pathlib import Path
import sys, subprocess, json
root = Path.cwd()
if not (root / 'vm_perceptual/infer.py').is_file():
    raise RuntimeError('请从 volcalmap 项目根目录运行')
module = root / 'vm_perceptual'
audio = module / 'data_jvs/jvs034/parallel100/wav24kHz16bit/VOICEACTRESS100_001.wav'
map_variant = 'current'  # current、student、rank_own 或 robust_student
display_dimensions = 2  # 可选 1、2、3
output = root / 'notebook_output'
output.mkdir(exist_ok=True)


In [ ]:
query_path = output / f'{map_variant}_query.json'
if map_variant == 'current':
    index_path = module / 'results/map_index.json'
    cmd = ['infer.py', str(audio), '--device', 'cuda', '--display-dimensions', str(display_dimensions)]
elif map_variant in ('student', 'robust_student'):
    index_name = 'wave_index.json' if map_variant == 'student' else 'robust_student_index.json'
    index_path = module / 'results' / index_name
    cmd = ['wave_map_cli.py', '--variant', map_variant, '--device', 'cuda', 'query', str(audio), '--display-dimensions', str(display_dimensions)]
elif map_variant == 'rank_own':
    index_path = module / 'results/rank_own_index.json'
    cmd = ['rank_map_cli.py', '--variant', 'own', 'query', str(audio), '--display-dimensions', str(display_dimensions)]
else:
    raise ValueError('请选择 current、student、rank_own 或 robust_student')
subprocess.run([sys.executable, str(module / cmd[0]), *cmd[1:], '--output', str(query_path)], check=True)
sys.path.insert(0, str(module))
from map_index import search
query = json.loads(query_path.read_text())
index = json.loads(index_path.read_text())
neighbors = search(index, query, k=5)['matches']
print('音色坐标', query['timbre']['vector'])
print('显示坐标', query['display']['coordinates'])
for rank, n in enumerate(neighbors, 1):
    print(rank, round(n['distance'], 4), n['record_id'])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
points = np.array([r['display'] for r in index['records']])
q = query['display']['coordinates']
near = np.array([n['display'] for n in neighbors])
fig = plt.figure(figsize=(8,5))
ax = fig.add_subplot(111, projection='3d' if display_dimensions == 3 else None)
def coords(arr):
    if display_dimensions == 1:
        return [arr[:,0], np.zeros(len(arr))]
    return [arr[:,j] for j in range(display_dimensions)]
ax.scatter(*coords(points), s=10, alpha=.35, label='Gallery')
ax.scatter(*coords(near), s=40, label='Full-space neighbors')
ax.scatter(*coords(np.array([q])), s=140, marker='*', label='Query')
ax.legend()
ax.set_xlabel('PC 1')
if display_dimensions > 1: ax.set_ylabel('PC 2')
if display_dimensions == 3: ax.set_zlabel('PC 3')
if display_dimensions == 1: ax.set_yticks([])
from io import BytesIO
from IPython.display import Image, display
buffer = BytesIO()
fig.savefig(buffer, format='png', bbox_inches='tight')
plt.close(fig)
display(Image(data=buffer.getvalue()))


星号是输入录音，橙色点是按完整 16 维距离找到的五个近邻。相同音频文件已从候选中排除。二维显示会压缩距离。下面可试听输入和首个近邻。

In [ ]:
from IPython.display import Audio, display
display(Audio(filename=str(audio)))
display(Audio(filename=str(module/'data_jvs'/neighbors[0]['record_id'])))


## 以当前录音为中心看近邻

上面的全局地图保留各录音的固定位置。下图将本次查询放在原点，每个点到原点的距离精确等于完整音色空间中的距离。编号对应检索排名。更换查询后，局部视图的方向可以改变。

In [ ]:
sys.path.insert(0, str(module))
from focus_view import focus
local = focus(index, query, k=8, dimensions=display_dimensions)
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='3d' if display_dimensions == 3 else None)
origin = np.zeros((1, display_dimensions))
ax.scatter(*coords(origin), s=130, marker='*', label='Query')
for rank, match in enumerate(local['neighbors'], 1):
    p = np.array([match['coordinates']])
    line = np.vstack((origin, p))
    ax.plot(*coords(line), color='0.8', linewidth=1)
    ax.scatter(*coords(p), s=30)
    ax.text(*[a[0] for a in coords(p)], str(rank))
    print(f"{rank}. {match['distance']:.3f}   {match['record_id']}")
if display_dimensions == 3:
    span = max((m['distance'] for m in local['neighbors']), default=1)
    for setter in (ax.set_xlim, ax.set_ylim, ax.set_zlim): setter(-span, span)
    ax.set_box_aspect((1,1,1))
else:
    ax.set_aspect('equal', adjustable='datalim')
    if display_dimensions == 1: ax.set_yticks([])
ax.set_title('Distances from the query')
ax.legend()
buffer = BytesIO()
fig.savefig(buffer, format='png', bbox_inches='tight')
plt.close(fig)
display(Image(data=buffer.getvalue()))
